# EET vs environmental drivers -- Sweden peatlands 2024

**Design:** 3 peatlands x 3 depths, one sample per cell = 9 samples, with
1:1 paired metagenomes (MG) and metatranscriptomes (MT).

**Question:** does extracellular electron transfer (EET) gene abundance
(MG) / expression (MT) relate to the CO2:CH4 ratio?

**Methodological notes (read before running):**
- At n=9 there is no replication within a Peatland x Depth cell, so no
  random-effects structure can be stimated -- exact permutation tests and
  bootstrapped confidence intervals are used instead of asymptotic ones.
- Sample names are used exactly as supplied in the metadata and coverage tables; no aliases or depth-name corrections are applied.


## 1. Import and prepare data

In [ ]:
# Load required packages
suppressPackageStartupMessages({
  library(tidyverse)   # dplyr, tibble, ggplot2, etc.
  library(reshape2)    # melt() for wide-to-long reshaping
  library(data.table)  # fast fread() for large coverage tables
  library(vegan)       # constrained ordination (RDA)
})

data.table::setDTthreads(1)
options(datatable.useThreads = FALSE)

# Input files
metadata_file <- "Sweden_peatlands_2024.spatial_sample_meta.txt"
mg_coverage_file <- "Sweden_peatlands_2024.mags.EET.MG_cov_normalised.tsv"
mt_coverage_file <- "Sweden_peatlands_2024.mags.EET.MT_cov_normalised.tsv"


In [ ]:
metadata <- read_tsv(metadata_file,
                      na = c("", "NA", "-")) %>%
  rename(Peatland = Location) %>%
  mutate(
    Peatland = factor(Peatland),
    Depth    = factor(Depth, levels = c("0-10", "20-30", "50-60")),
    across(c(pH, Temperature, Electrical_conductivity, CO2_CH4, CH4, CO2,
             Nitrate, Sulfate, Total_Mn, Total_Fe, Propionate, Acetate),
           as.numeric)
  )

eet_mg_coverage <- fread(mg_coverage_file, sep = "\t", header = TRUE)
eet_mt_coverage <- fread(mt_coverage_file, sep = "\t", header = TRUE)

In [ ]:
# Reshape to long format and restrict to the final 9 samples
genes_mg <- eet_mg_coverage %>%
  reshape2::melt(id.vars = c("Protein_ID", "EET_type"), variable.name = "Sample", value.name = "MG_coverage_per_cell") %>%
  unique() %>%
  filter(Sample %in% metadata$Sample_name_MG)

genes_mt <- eet_mt_coverage %>%
  reshape2::melt(id.vars = c("Protein_ID", "EET_type"), variable.name = "Sample", value.name = "MT_coverage_per_cell") %>%
  unique() %>%
  filter(Sample %in% metadata$Sample_name_MT)

# Sanity check -- both counts should read 9, and both setdiffs empty
cat("MG samples retained:", n_distinct(genes_mg$Sample), "(expect 9)\n")
cat("MT samples retained:", n_distinct(genes_mt$Sample), "(expect 9)\n")
setdiff(metadata$Sample_name_MG, unique(genes_mg$Sample))
setdiff(metadata$Sample_name_MT, unique(genes_mt$Sample))

## 2. Aggregate (per-sample) EET score vs CO2:CH4 ratio

Sum EET gene coverage into one score per sample, then test that score
against the CO2:CH4 ratio using an **exact within-peatland permutation
test** (216 possible relabelings) and a **bootstrapped effect-size CI**.

In [ ]:
# Sum EET gene coverage per sample: MG = genomic abundance, MT = expression
eet_sample_mg <- genes_mg %>%
  group_by(Sample) %>%
  summarise(MG_total = sum(MG_coverage_per_cell, na.rm = TRUE),
            n_genes_MG_detected = sum(MG_coverage_per_cell > 0),
            .groups = "drop")

eet_sample_mt <- genes_mt %>%
  group_by(Sample) %>%
  summarise(MT_total = sum(MT_coverage_per_cell, na.rm = TRUE),
            n_genes_MT_detected = sum(MT_coverage_per_cell > 0),
            .groups = "drop")

dat <- metadata %>%
  left_join(eet_sample_mg, by = c("Sample_name_MG" = "Sample")) %>%
  left_join(eet_sample_mt, by = c("Sample_name_MT" = "Sample"))

stopifnot(nrow(dat) == 9, all(!is.na(dat$MG_total)), all(!is.na(dat$MT_total)))
print(dat %>% select(Peatland, Depth, MG_total, MT_total, CO2, CH4, CO2_CH4))

In [ ]:
# Quick visual check of the score distributions -- at n=9 a formal
# normality test isn't meaningful, this just rules out a pathological
# single-outlier situation before running the rank-based tests below
hist(dat$MG_total, breaks = 6, main = "MG_total (raw, per-cell coverage)")
hist(dat$MT_total, breaks = 6, main = "MT_total (raw, per-cell coverage)")

In [ ]:
predictor <- "CO2_CH4"

##### 
# Exact within-peatland permutation test 
#####
## Enumerates all (3!)^3 = 216 within-block relabelings, giving an exact
## null distribution appropriate for n=9 (rather than an asymptotic p-value)
all_perms <- function(n) {
  if (n == 1) return(matrix(1, 1, 1))
  sub <- all_perms(n - 1)
  do.call(rbind, lapply(1:n, function(i) cbind(i, sub + (sub >= i))))
}

exact_block_permutation_test <- function(data, response, predictor, block = "Peatland") {
  data <- data %>% arrange(.data[[block]])
  block_sizes <- table(data[[block]])
  stopifnot(length(unique(block_sizes)) == 1)
  block_perms <- all_perms(unique(block_sizes))
  block_ids   <- unique(data[[block]])
  perm_grid   <- expand.grid(rep(list(seq_len(nrow(block_perms))), length(block_ids)))

  obs_stat <- cor(data[[response]], data[[predictor]], method = "spearman")
  perm_stats <- apply(perm_grid, 1, function(idx) {
    permuted <- data[[response]]
    for (b in seq_along(block_ids)) {
      rows <- which(data[[block]] == block_ids[b])
      permuted[rows] <- data[[response]][rows][block_perms[idx[b], ]]
    }
    cor(permuted, data[[predictor]], method = "spearman")
  })

  tibble(predictor = predictor, rho = obs_stat,
         p_exact = mean(abs(perm_stats) >= abs(obs_stat)),
         n_permutations = length(perm_stats))
}

mg_perm <- exact_block_permutation_test(dat, "MG_total", predictor) %>% mutate(omic = "MG")
mt_perm <- exact_block_permutation_test(dat, "MT_total", predictor) %>% mutate(omic = "MT")
print(bind_rows(mg_perm, mt_perm))

##### 
# Effect size with bootstrap confidence interval
#####
## A p-value alone is uninformative at n=9 -- report rho with a CI so a
## null result reads as "wide uncertainty", not "no relationship"
boot_rho_ci <- function(data, response, predictor, n_boot = 2000) {
  obs <- cor(data[[response]], data[[predictor]], method = "spearman")
  boots <- replicate(n_boot, {
    idx <- sample(seq_len(nrow(data)), replace = TRUE)
    cor(data[[response]][idx], data[[predictor]][idx], method = "spearman")
  })
  tibble(predictor = predictor, rho = obs,
         ci_low  = quantile(boots, 0.025, na.rm = TRUE),
         ci_high = quantile(boots, 0.975, na.rm = TRUE))
}

mg_ci <- boot_rho_ci(dat, "MG_total", predictor) %>% mutate(omic = "MG")
mt_ci <- boot_rho_ci(dat, "MT_total", predictor) %>% mutate(omic = "MT")
print(bind_rows(mg_ci, mt_ci))

In [ ]:
# Aggregate EET score vs CO2:CH4 ratio (visual companion to the tests above)
ggplot(dat, aes(x = CO2_CH4, y = MG_total, color = Peatland, shape = Depth)) +
  geom_point(size = 3) +
  geom_smooth(method = "lm", se = FALSE, aes(group = 1), color = "black", linetype = "dashed") +
  labs(x = "CO2:CH4 ratio", y = "Total EET MG coverage per cell",
       title = "EET gene abundance vs CO2:CH4 ratio (n = 9)") +
  theme_minimal()

ggplot(dat, aes(x = CO2_CH4, y = MT_total, color = Peatland, shape = Depth)) +
  geom_point(size = 3) +
  geom_smooth(method = "lm", se = FALSE, aes(group = 1), color = "black", linetype = "dashed") +
  labs(x = "CO2:CH4 ratio", y = "Total EET MT coverage per cell",
       title = "EET gene expression vs CO2:CH4 ratio (n = 9)") +
  theme_minimal()

## 3. Gene-level community analysis (constrained ordination)

Rather than collapsing all genes into one score, this treats the EET
genes as a community matrix (samples x genes) and asks whether CO2:CH4
explains a coherent axis of variation -- this can detect structure even
when individual genes move in different directions and cancel out in
the aggregate score above.

In [ ]:
build_gene_matrix <- function(genes, value_col, sample_order) {
  genes %>%
    select(Sample, Protein_ID, all_of(value_col)) %>%
    pivot_wider(names_from = Protein_ID, values_from = all_of(value_col),
                values_fill = 0, values_fn = sum) %>%
    column_to_rownames("Sample") %>%
    .[sample_order, ]
}

mg_mat <- build_gene_matrix(genes_mg, "MG_coverage_per_cell", dat$Sample_name_MG)
mt_mat <- build_gene_matrix(genes_mt, "MT_coverage_per_cell", dat$Sample_name_MT)

# Hellinger transform
mg_mat_h <- decostand(mg_mat, method = "hellinger")
mt_mat_h <- decostand(mt_mat, method = "hellinger")

# Restricted permutation: only shuffle within Peatland (216 possible = (3!)^3)
ctrl <- how(blocks = dat$Peatland, nperm = 215)

run_single_rda <- function(mat_h, predictor, env, ctrl) {
  m <- rda(as.formula(paste0("mat_h ~ ", predictor)), data = env)
  list(model = m, anova = anova(m, permutations = ctrl, by = "terms"))
}

env <- dat %>% select(CO2_CH4)
mg_rda <- run_single_rda(mg_mat_h, predictor, env, ctrl)
mt_rda <- run_single_rda(mt_mat_h, predictor, env, ctrl)

print(mg_rda$anova)
print(mt_rda$anova)

# BH correction across the 2 tests (MG, MT)
rda_all <- tibble(
  omic    = c("MG", "MT"),
  F_value = c(mg_rda$anova$F[1], mt_rda$anova$F[1]),
  p_value = c(mg_rda$anova$`Pr(>F)`[1], mt_rda$anova$`Pr(>F)`[1])
) %>% mutate(p_adj = p.adjust(p_value, method = "BH"))
print(rda_all)

## 4. Which EET type drives the CO2:CH4 axis?

Tests whether the constrained axis above is driven disproportionately
by one of the 4 EET types. Uses the **genes represented in the input tables** as the unit of comparison rather than the 9 samples.

In [ ]:
gene_type_lookup <- genes_mg %>%
  mutate(EET_type = gsub("_.*", "", EET_type)) %>%
  distinct(Protein_ID, EET_type)

mg_scores_full <- scores(mg_rda$model, display = "species", choices = 1) %>%
  as.data.frame() %>% rownames_to_column("Protein_ID") %>%
  left_join(gene_type_lookup, by = "Protein_ID")

mt_scores_full <- scores(mt_rda$model, display = "species", choices = 1) %>%
  as.data.frame() %>% rownames_to_column("Protein_ID") %>%
  left_join(gene_type_lookup, by = "Protein_ID")

ggplot(mg_scores_full, aes(x = EET_type, y = RDA1, fill = EET_type)) +
  geom_boxplot(alpha = 0.6) + geom_jitter(width = 0.15) +
  labs(title = "MG: gene loadings on CO2:CH4 axis by EET_type") +
  theme_minimal() + theme(legend.position = "none")

ggplot(mt_scores_full, aes(x = EET_type, y = RDA1, fill = EET_type)) +
  geom_boxplot(alpha = 0.6) + geom_jitter(width = 0.15) +
  labs(title = "MT: gene loadings on CO2:CH4 axis by EET_type") +
  theme_minimal() + theme(legend.position = "none")

# Does loading magnitude differ by EET_type? (2 tests total, genes as the unit)
kruskal.test(abs(RDA1) ~ EET_type, data = mg_scores_full)
kruskal.test(abs(RDA1) ~ EET_type, data = mt_scores_full)

## 5. Per-gene directionality check

Descriptive only, **not** a per-gene significance test (n=9 gives
essentially no power per gene). Checks whether the null aggregate score
above reflects genuinely flat genes, or genes with real but opposing
relationships that cancel out when summed.

In [ ]:
gene_rho_distribution <- function(mat, env_var, gene_type_lookup) {
  rhos <- apply(mat, 2, function(col) {
    if (sd(col) == 0) return(NA_real_)  # constant across samples -- undefined
    suppressWarnings(cor(col, env_var, method = "spearman"))
  })
  tibble(Protein_ID = names(rhos), rho = rhos) %>%
    filter(!is.na(rho)) %>%
    left_join(gene_type_lookup, by = "Protein_ID")
}

mg_gene_rhos <- gene_rho_distribution(mg_mat, dat$CO2_CH4, gene_type_lookup) %>% mutate(omic = "MG")
mt_gene_rhos <- gene_rho_distribution(mt_mat, dat$CO2_CH4, gene_type_lookup) %>% mutate(omic = "MT")

# A roughly even positive/negative split centred near zero indicates
# cancelling/opposing patterns; a lopsided split with weak individual
# rhos indicates a diluted-but-consistent effect instead
bind_rows(mg_gene_rhos, mt_gene_rhos) %>%
  group_by(omic) %>%
  summarise(n_genes = n(), median_rho = median(rho),
            pct_positive = mean(rho > 0) * 100,
            pct_negative = mean(rho < 0) * 100,
            .groups = "drop") %>%
  print()

ggplot(bind_rows(mg_gene_rhos, mt_gene_rhos), aes(x = rho)) +
  geom_histogram(bins = 30, fill = "steelblue", alpha = 0.7) +
  geom_vline(xintercept = 0, linetype = "dashed") +
  facet_wrap(~omic) +
  labs(title = "Per-gene Spearman rho vs CO2:CH4 ratio",
       x = "rho (individual EET gene vs CO2:CH4)", y = "Number of genes") +
  theme_minimal()